# Trajectory analysis of a replayable hierarchy run

Choose the run with `RUN_NAME` in the first code cell (any folder under `results/designed_init/` that holds a
`replay/` sub-folder written by `save_designed_run`). Run 11 (`hierarchy_replayable`) trains the run-10 design (80-30-40; balanced output, path-sign-aware cues, equalised cue blocks,
balanced +-10-quanta third-layer paths; cues frozen) on 1M training points with batch 10 000, so one
epoch is 100 steps; 10 000 epochs (1M steps), lr 0.01 annealed by cosine to 0.0005, momentum 0.9.
Weights **and momentum buffers** are stored every 10 epochs (1000 steps), so the training between any
two adjacent checkpoints can be replayed exactly, step by step, with the mini-batch gradient, the update
and (optionally) the population gradient of every step. Run 12 (`hierarchy_replayable_probe001`) is run 11 with the 30 probe rows multiplied by 0.01 at init. Files:
`results/designed_init/<RUN_NAME>/replay/` (`weights.npz`, `momentum.npz`, `meta.pkl`; loader `load_designed_run`),
produced by `results/designed_init/run_<RUN_NAME>.py`.

Tools: `replay_interval_from_snapshot` (exact replay between checkpoints, per-step records),
`population_gradient_at_snapshot`, `first_layer_gradient_series` / `plot_first_layer_gradient_series`
(components along ODT nodes and the off-span norm), `first_layer_alignment_series` /
`plot_first_layer_alignment_series`, `probe_gradient_stats`, `hierarchy_run_summary`.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
%load_ext autoreload
%autoreload 2
from layer_lenses.designed_init import HIERARCHY_TARGETS, NODE_LEVEL, hierarchy_run_summary, probe_node_outcomes, plot_probe_target_cosines, probe_gradient_stats
from layer_lenses.relu_analysis import (load_designed_run, regenerate_seed_result_data, replay_interval_from_snapshot, population_gradient_at_snapshot,
                                         first_layer_gradient_series, plot_first_layer_gradient_series, first_layer_alignment_series, plot_first_layer_alignment_series,
                                         snapshot_state_dict, first_layer_odt_alignment, plot_layered_relu_graph,
                                         replay_steps_full, loss_on_points, loss_along_update, layerwise_update_ablation, hessian_quadratic_form, top_hessian_eigenvalue, flatten_parameter_dict)

RESULTS = Path("../results/designed_init")
available = sorted(d.name for d in RESULTS.iterdir() if (d / "replay" / "meta.pkl").exists())
print("replayable runs:", available)
RUN_NAME = "hierarchy_scalaradam_lr03"          # run 11 is hierarchy_replayable ; run 12 is "hierarchy_replayable_probe001"(probe rows x 0.01 at init)
RUN_DIR = RESULTS / RUN_NAME
sr = load_designed_run(RUN_DIR / "replay")
tree = sr["tree"]; cfg = sr["train_config"]; out = sr["out"]
data = regenerate_seed_result_data(sr)                      # 1M training points + 200k test points, from their seeds (~3 s)
epochs = np.array(sorted(out["checkpoint_snapshots"])); steps_per_epoch = data["x_train"].shape[0] // cfg.batch_size
W = {k: np.stack([out["checkpoint_snapshots"][e][k].numpy() for e in epochs]) for k in out["checkpoint_snapshots"][epochs[0]]}
run = {"tag": sr["tag"], "epochs": epochs, "W": W, "losses": np.array(out["epoch_losses"]), "lr": np.array(out["lr_history"])}
print(f"{sr['tag']}: {len(epochs)} snapshots every {epochs[1]} epochs ({epochs[1] * steps_per_epoch} steps); {cfg.epochs} epochs x {steps_per_epoch} steps; design flipped units {sr['design']['flipped_units']}")


## 1. Overview: loss, probe cosines to their targets, node coverage

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
axes[0].plot(np.arange(1, len(run["losses"]) + 1) * steps_per_epoch, run["losses"], lw=0.6); axes[0].set_yscale("log"); axes[0].set_xlabel("step"); axes[0].set_ylabel("epoch (100-step) train loss")
cov = [hierarchy_run_summary(W["hidden_layers.0.weight"][i], tree)["nodes_covered"] for i in range(len(epochs))]
axes[1].plot(epochs * steps_per_epoch, cov); axes[1].axhline(31, color="k", ls=":", lw=0.8); axes[1].set_xlabel("step"); axes[1].set_ylabel("nodes covered (|cos| > 0.95)")
axes[2].plot(np.arange(1, len(run["lr"]) + 1) * steps_per_epoch, run["lr"]); axes[2].set_xlabel("step"); axes[2].set_ylabel("learning rate")
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()
plot_probe_target_cosines(W["hidden_layers.0.weight"], epochs, tree, title=f"{sr['tag']}: signed cosine of each probe to its designed target (x = epoch of 100 steps)"); plt.show()
s_end = hierarchy_run_summary(W["hidden_layers.0.weight"][-1], tree); print({k: v for k, v in s_end.items() if k != "missing_nodes"})
display(probe_node_outcomes(W["hidden_layers.0.weight"][-1], tree))


## 2. Exact replay of one checkpoint interval

`replay_interval_from_snapshot` restarts from the stored weights and momentum at `EPOCH_START`, replays the
1000 steps to the next checkpoint with the stored learning rates and the fixed batch order, and checks that it
lands exactly on the next snapshot. Per step it records the batch loss, the mini-batch gradient and the update of
the chosen first-layer rows, and the population gradient on those rows over the first `FULL_SUBSAMPLE` training
points (the whole 1M set if None; 200k costs ~20 ms per step, 1M ~100 ms).


In [ ]:
EPOCH_START = 1750          # any stored snapshot epoch except the last (multiples of 10)
ROWS = [0, 1, 2, 3, 4, 5,30]    # first-layer rows to record (the u_0, u_1, u_2 probe pairs)
FULL_SUBSAMPLE = 200_000     # points for the per-step population gradient; None = all 1M

rep = replay_interval_from_snapshot(sr, EPOCH_START, data=data, device="cuda", record_full_data_gradient=True, full_data_subsample=FULL_SUBSAMPLE, record_params=True, neuron_rows=ROWS)
print(f"replayed epochs {rep['epoch_start']}..{rep['epoch_end']} = {rep['num_steps']} steps; exact to the next snapshot: max |param diff| {rep['verification']['max_abs_param_diff_overall']:.2e}, max |momentum diff| {rep['verification']['max_abs_momentum_diff']:.2e}")


In [ ]:
print(rep.keys())
print(rep["params_rows"].shape)

## 2b. The recorded rows step by step within the interval

`rep["params_rows"]` holds the recorded first-layer rows before every step of the replayed interval. For each row:
cosine to the chosen ODT nodes and the row norm (left panels, from `first_layer_alignment_series`), and the size
of each step's movement, split into the part along the row (norm change) and the part perpendicular to it
(rotation), together with the mini-batch gradient norm (right panels). Set `NODE_IDS_P = None` to use each row's
designed node and the node it is closest to at the end of the interval.


In [ ]:
# params_rows within the interval: cosines to ODT nodes, row norms, and per-step movement of the recorded rows
NODE_IDS_P = None            # list of ODT nodes, or None for designed + closest-at-end nodes of the recorded rows
ROWS_P = ROWS                # subset of the rows recorded in the replay cell

P = rep["params_rows"]; n_steps, _, dim = P.shape; n_rows = W["hidden_layers.0.weight"].shape[1]
Pfull = np.zeros((n_steps, n_rows, dim))
for j, r in enumerate(rep["rows"]):
    Pfull[:, r] = P[:, j]
if NODE_IDS_P is None:
    C_end = (Pfull[-1] / np.maximum(np.linalg.norm(Pfull[-1], axis=1, keepdims=True), 1e-12)) @ tree.w_list.T
    NODE_IDS_P = sorted({HIERARCHY_TARGETS[r][0] for r in ROWS_P if r < 30} | {int(np.abs(C_end[r]).argmax()) for r in ROWS_P})
aser = first_layer_alignment_series(Pfull, rep["step"], tree, neuron_ids=ROWS_P, node_ids=NODE_IDS_P)
fig, axes = plot_first_layer_alignment_series(aser, title=f"rows {ROWS_P} vs ODT nodes {NODE_IDS_P}, steps of the replayed interval (from epoch {EPOCH_START})", show_fan_out=False)
axes[-1].set_xlabel(f"step within the interval"); plt.show()

fig, axes = plt.subplots(len(ROWS_P), 1, figsize=(9, 2.4 * len(ROWS_P)), sharex=True, squeeze=False); axes = axes[:, 0]
for ax, r in zip(axes, ROWS_P):
    j = rep["rows"].index(r); w = P[:, j]; what = w / np.maximum(np.linalg.norm(w, axis=1, keepdims=True), 1e-12); dw = rep["update_rows"][:, j]
    along = np.einsum("td,td->t", dw, what); perp = np.linalg.norm(dw - along[:, None] * what, axis=1)
    ax.plot(rep["step"], np.abs(along), lw=0.9, label="|step along the row| (norm change)"); ax.plot(rep["step"], perp, lw=0.9, label="|step perpendicular to the row| (rotation)")
    ax.plot(rep["step"], np.linalg.norm(rep["grad_rows"][:, j], axis=1), lw=0.7, ls=":", color="gray", label="mini-batch gradient norm")
    ax.set_yscale("log"); ax.set_ylabel("per-step size"); ax.set_title(f"row {r} ({'+' if HIERARCHY_TARGETS[r][1] > 0 else '-'}u{HIERARCHY_TARGETS[r][0]})" if r < 30 else f"row {r}", fontsize=9); ax.grid(alpha=0.25); ax.legend(fontsize=6, ncol=3)
axes[-1].set_xlabel("step within the interval"); fig.tight_layout(); plt.show()
aser = aser.rename(columns={"epoch": "step"})          # the series tool names its time column "epoch"; here it is the step
display(aser[aser.step.isin([0, 250, 500, 750, n_steps - 1])].round(4).set_index(["neuron", "step"]))


## 3. Per-step population gradient on chosen rows vs step

Components of the per-step push (-grad) along chosen ODT nodes and the off-span norm, for the rows recorded above.
The step-level view shows what the 10-epoch snapshots cannot: whether the push is a steady drift or an oscillation
whose phase the snapshots happen to sample.


In [ ]:
NODE_IDS = list(range(31))  # all nodes in the hierarchy
PUSH, NORMALIZE = True, False
grads_by_step = {int(s): rep["full_grad_rows"][i] for i, s in enumerate(rep["step"])}
# the series tool expects full first-layer gradients; embed the recorded rows into a zero matrix of the right width
n_rows, dim = W["hidden_layers.0.weight"].shape[1:]
full = {s: np.zeros((n_rows, dim)) for s in grads_by_step}
for s, g in grads_by_step.items():
    full[s][ROWS] = g
gser = first_layer_gradient_series(full, tree, neuron_ids=ROWS, node_ids=NODE_IDS, push=PUSH, normalize=NORMALIZE).rename(columns={"epoch": "step"})
fig, axes = plt.subplots(len(ROWS), 1, figsize=(10, 2.6 * len(ROWS)), sharex=True, squeeze=False); axes = axes[:, 0]
for ax, r in zip(axes, ROWS):
    g = gser[gser.neuron == r]
    for k, n in enumerate(NODE_IDS): ax.plot(g.step, g[f"comp_{n}"], lw=0.8, label=f"along u_{n}")
    ax.plot(g.step, g.inspan_norm, color="k", lw=0.8, ls="--", label="in-span norm");
    ax.plot(g.step, g.offspan_norm, color="r", lw=0.8, ls="--", label="off-span norm"); 
    ax.plot(g.step, g.grad_norm, color="gray", lw=0.6, ls=":", label="total norm")
    ax.axhline(0, color="gray", lw=0.5); ax.set_ylabel(("cos " if NORMALIZE else "") + ("push" if PUSH else "grad")); ax.set_title(f"row {r} ({'+' if HIERARCHY_TARGETS[r][1] > 0 else '-'}u{HIERARCHY_TARGETS[r][0]})" if r < 30 else f"row {r} (cue)" if r < 62 else f"row {r} (random)", fontsize=9); ax.grid(alpha=0.25); ax.legend(fontsize=6, ncol=5)
axes[-1].set_xlabel(f"step within the replayed interval (from epoch {EPOCH_START})"); fig.tight_layout(); plt.show()
# running mean of the push vs the per-step push: what the row actually integrates
for r in [r for r in ROWS if r < 30]:          # the target-relative numbers only make sense for probes
    g = gser[gser.neuron == r]; mean_push = -rep["full_grad_rows"][:, ROWS.index(r)].mean(0); U0 = tree.w_list[HIERARCHY_TARGETS[r][0]] / np.linalg.norm(tree.w_list[HIERARCHY_TARGETS[r][0]])
    print(f"row {r}: mean per-step |push| {g.grad_norm.mean():.4f}, |mean push| {np.linalg.norm(mean_push):.4f}, mean push along its target {HIERARCHY_TARGETS[r][1] * (mean_push @ U0):+.4f}, actual displacement over the interval along target {HIERARCHY_TARGETS[r][1] * ((rep['params_rows'][-1, ROWS.index(r)] - rep['params_rows'][0, ROWS.index(r)]) @ U0):+.4f}")


## 4. Mini-batch step vs population gradient, and the batch loss, within the interval

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(rep["step"], rep["batch_loss"], lw=0.6); axes[0].set_ylabel("mini-batch loss"); axes[0].set_xlabel("step")
cosines = [float((-rep["grad_rows"][i, 0] @ -rep["full_grad_rows"][i, 0]) / (np.linalg.norm(rep["grad_rows"][i, 0]) * np.linalg.norm(rep["full_grad_rows"][i, 0]) + 1e-30)) for i in range(rep["num_steps"])]
axes[1].plot(rep["step"], cosines, lw=0.6); axes[1].set_ylabel(f"cos(mini-batch grad, population grad), row {ROWS[0]}"); axes[1].set_xlabel("step"); axes[1].set_ylim(-1.05, 1.05)
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()


## 5. Anatomy of a loss spike: what the steps before a blow-up do

`replay_steps_full` replays an interval from its snapshot and records, for a window of steps, *every* parameter
tensor before the step, the SGD momentum buffers, the mini-batch gradient, the update, the batch loss and the loss on
a fixed probe set (the first `PROBE_POINTS` training points, so a loss rise can be told apart from a hard batch).
On top of it: `top_hessian_eigenvalue` (power iteration on the probe points, restricted to the trainable
coordinates), `hessian_quadratic_form` (curvature along a given direction), `loss_along_update` (line search along
the step the optimizer took) and `layerwise_update_ablation` (the step applied to one tensor only / all but one).

Heavy-ball SGD with momentum beta is linearly stable along a Hessian eigendirection iff `lr * lambda < 2 (1 + beta)`
(3.8 here); above that the mode grows with a sign flip every step. For a ScalarAdam run the cells use the SGD-equivalent
learning rate `lr (1 - beta1) / ((1 - beta1^t) sqrt(v_hat))` (`anat["lr_sgd_equiv"]`) and beta1, so the same products apply;
for AdamW there is no scalar equivalent and the stability columns are NaN. The reference window is run 11's epoch 1759, whose
last step (step 999 of the interval from 1750) sees a batch loss of 2.2.

**Findings for run 11, steps 980-999 of the interval from epoch 1750.**
- Not a bad batch: the probe loss tracks the batch loss (0.83 -> 2.28 -> 5.64 over steps 998, 999 and after).
- Not one bad step: a six-step geometric growth. From step 994 the update norm grows x1.3-1.7 per step
  (0.042 -> 0.28), consecutive gradients are anti-parallel (cos -0.92 to -0.99) and so are consecutive updates
  (cos -0.85 to -0.91): a period-2 oscillation along one direction with growing amplitude. The gradient norm at
  step 994 (5.7) is ordinary for this epoch; what matters is that it alternates sign along a direction whose
  curvature is past the stability limit.
- The direction is the top Hessian eigenvector (cos(update, eigenvector) 0.92-0.98 from step 994), 59% of it in W2
  and 40% in W3, under 1% in the first layer and the output.
- Stability: through steps 980-993 `lr * lambda_max` hovers at 3.5-3.7 (mode multiplier 0.95, marginal); at step
  994 it crosses to 4.3 and reaches 5.1-5.3 at steps 996-998 (linear growth 2-3 per step; the observed 1.3-1.7 is
  smaller because ReLU gates cap the quadratic picture). The curvature along the update itself goes 3.7 -> 5.1 in
  lr units.
- Why it crosses: the oscillation flips layer-3 gates (0.05% -> 0.5% of (point, unit) gates per step as the
  amplitude grows) and every flip changes the local Hessian, so the sharpness rises with the amplitude. Once the
  mode is marginal this feedback lets it run away; progressive sharpening keeps it marginal all epoch.
- One step in detail (step 998): the loss along the update is 0.83 (alpha 0) -> 0.19 (alpha 0.5) -> 2.28 (alpha 1):
  the step is twice too long. Only W3's update: 0.13; only W2's: 0.36; both: 2.28. Each layer's update alone would
  have helped; their sum overshoots because both move the same valley direction (the product W3 W2).
- The momentum buffer is not the driver: at step 998 the buffer term of the update (0.095) opposes the gradient term
  (0.25), cos -0.96, because the buffer still carries the previous half-period. The growth is in the gradient itself,
  which doubles each step as the oscillation amplitude doubles.
- Recovery is a catapult: the loss keeps flipping (5.6, 2.2, 0.7, 2.3, 0.8, 1.5, ...) while `lr * lambda_max`
  collapses from 3.6 to 0.9 at the first step of epoch 1760 and stays at 2.2-2.5 for the next 40 steps; the loss is
  back to 0.15 within 40 steps. The big steps moved the iterate to a flatter region. Over the whole spike the net
  displacement is 0.92, of which W1 contributes 0.63 (its gradient is 7x its normal size at the peak): the first
  layer is kicked by the spike more than it moves in a normal epoch, which is where the "first-layer motion
  correlates with spikes" finding partly comes from.


In [ ]:
SPIKE_EPOCH_START = 1750       # snapshot epoch the window is replayed from (run 11: the epoch-1759 spike)
STEP_FROM, STEP_TO = 900, 1000  # steps within the interval (1000 per interval); the last recorded step is STEP_TO - 1
PROBE_POINTS = 20_000           # fixed training points for the probe loss, curvature and line searches
anat = replay_steps_full(sr, SPIKE_EPOCH_START, step_from=STEP_FROM, step_to=STEP_TO, data=data, device="cuda", probe_points=PROBE_POINTS)   # ~3 s
names = anat["parameter_names"]; tmpl = out["model"]; x_probe, y_probe = data["x_train"][:PROBE_POINTS], data["y_train"][:PROBE_POINTS]
short = lambda nm: {"hidden_layers.0.weight": "W1", "hidden_layers.1.weight": "W2", "hidden_layers.2.weight": "W3", "output_layer.weight": "Wout"}.get(nm, nm)
Uf = np.stack([flatten_parameter_dict(u, names) for u in anat["updates"]]); Gf = np.stack([flatten_parameter_dict(g, names) for g in anat["grads"]])
un = {short(nm): np.array([np.linalg.norm(u[nm]) for u in anat["updates"]]) for nm in names}; gn = {short(nm): np.array([np.linalg.norm(g[nm]) for g in anat["grads"]]) for nm in names}
cosU = np.r_[np.nan, (Uf[1:] * Uf[:-1]).sum(1) / (np.linalg.norm(Uf[1:], axis=1) * np.linalg.norm(Uf[:-1], axis=1))]; cosG = np.r_[np.nan, (Gf[1:] * Gf[:-1]).sum(1) / (np.linalg.norm(Gf[1:], axis=1) * np.linalg.norm(Gf[:-1], axis=1))]
st = anat["step"]
fig, axes = plt.subplots(2, 2, figsize=(13, 6.5), sharex=True)
axes[0, 0].plot(st, anat["batch_loss"], lw=0.8, label="mini-batch loss"); axes[0, 0].plot(st, anat["probe_loss"], lw=0.8, ls="--", label=f"loss on {PROBE_POINTS} fixed points"); axes[0, 0].set_yscale("log"); axes[0, 0].legend(fontsize=7)
for k, v in un.items(): axes[0, 1].plot(st, v, lw=0.8, label=f"|update {k}|")
axes[0, 1].set_yscale("log"); axes[0, 1].legend(fontsize=7, ncol=2)
for k, v in gn.items(): axes[1, 0].plot(st, v, lw=0.8, label=f"|grad {k}|")
axes[1, 0].set_yscale("log"); axes[1, 0].legend(fontsize=7, ncol=2); axes[1, 0].set_xlabel("step within the interval")
axes[1, 1].plot(st, cosU, lw=0.8, label="cos(update_t, update_t-1)"); axes[1, 1].plot(st, cosG, lw=0.8, label="cos(grad_t, grad_t-1)"); axes[1, 1].axhline(0, color="gray", lw=0.5); axes[1, 1].legend(fontsize=7); axes[1, 1].set_xlabel("step within the interval")
for ax in axes.ravel(): ax.grid(alpha=0.25)
fig.suptitle(f"{sr['tag']}: steps {STEP_FROM}-{STEP_TO - 1} of the interval from epoch {SPIKE_EPOCH_START}", fontsize=10); fig.tight_layout(); plt.show()
tab = pd.DataFrame({"step": st, "epoch": anat["epoch"], "batch": anat["batch_index"], "batch_loss": anat["batch_loss"], "probe_loss": anat["probe_loss"], "|update|": np.linalg.norm(Uf, axis=1), "|grad|": np.linalg.norm(Gf, axis=1),
                    "cos(upd,prev)": cosU, "cos(grad,prev)": cosG, **{f"|upd {k}|": v for k, v in un.items()}})
print(tab.tail(12).round(4).to_string(index=False)); print("loss on the probe points after the last recorded step:", round(anat["probe_loss_after"], 4))

In [ ]:
# Sharpness along the window: top Hessian eigenvalue on the probe points (restricted to the trainable coordinates), the heavy-ball
# stability limit 2(1+beta), the curvature along each step's own update, and how aligned the update is with the top eigenvector.
SHARP_STEPS = list(range(STEP_TO - 20, STEP_TO))          # steps (within the interval) to evaluate; ~0.3 s each
beta = anat["beta1"]; lr_eq = anat["lr_sgd_equiv"]; mask_np = {nm: np.asarray(cfg.parameter_update_mask[nm]) for nm in names} if cfg.parameter_update_mask is not None else None
def heavy_ball_multiplier(lr_lam):                        # largest |root| of z^2 - (1 + beta - lr*lambda) z + beta = 0: the mode grows iff > 1
    return float(np.abs(np.roots([1.0, -(1.0 + beta - lr_lam), beta])).max())
rows = []
for s_ in SHARP_STEPS:
    i = int(np.nonzero(st == s_)[0][0]); p = anat["params_before"][i]; lr = lr_eq[i]           # SGD-equivalent learning rate of this step
    ev = top_hessian_eigenvalue(tmpl, p, x_probe, y_probe, device="cuda", iters=25, mask=mask_np); q = hessian_quadratic_form(tmpl, p, anat["updates"][i], x_probe, y_probe, device="cuda")
    rows.append({"step": s_, "batch_loss": anat["batch_loss"][i], "|update|": np.linalg.norm(Uf[i]), "lr_eq*lambda_max": ev["lambda_max"] * lr, "mode multiplier": heavy_ball_multiplier(ev["lambda_max"] * lr) if np.isfinite(lr) else np.nan,
                 "lr_eq*curv(update)": q["curvature"] * lr, "cos(update, eigvec)": abs(float(flatten_parameter_dict(ev["eigenvector"], names) @ Uf[i] / np.linalg.norm(Uf[i]))), **{f"eigvec frac {short(k)}": v for k, v in ev["layer_fraction"].items()}})
sharp = pd.DataFrame(rows); print(sharp.round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(sharp.step, sharp["lr_eq*lambda_max"], marker="o", ms=3, label="lr_eq * lambda_max"); axes[0].plot(sharp.step, sharp["lr_eq*curv(update)"], marker="s", ms=3, label="lr_eq * curvature along the update"); axes[0].axhline(2 * (1 + beta), color="r", ls="--", lw=0.8, label=f"2(1+beta) = {2 * (1 + beta):.1f}"); axes[0].legend(fontsize=7)
axes[1].plot(sharp.step, sharp["mode multiplier"], marker="o", ms=3, label="heavy-ball mode multiplier at lambda_max"); axes[1].plot(sharp.step[1:], np.linalg.norm(Uf[[int(np.nonzero(st == s_)[0][0]) for s_ in SHARP_STEPS]], axis=1)[1:] / np.linalg.norm(Uf[[int(np.nonzero(st == s_)[0][0]) for s_ in SHARP_STEPS]], axis=1)[:-1], marker="s", ms=3, label="observed |update_t| / |update_t-1|"); axes[1].axhline(1, color="r", ls="--", lw=0.8); axes[1].legend(fontsize=7)
for ax in axes: ax.set_xlabel("step within the interval"); ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()

In [ ]:
# One step in detail: the loss along the update the optimizer took (alpha = 1), and the update applied to one tensor at a time.
STEP_FOCUS = STEP_TO - 2        # the step whose update produces the recorded blow-up (its batch loss is seen at STEP_FOCUS + 1)
i = int(np.nonzero(st == STEP_FOCUS)[0][0]); p, u = anat["params_before"][i], anat["updates"][i]
line = loss_along_update(tmpl, p, u, x_probe, y_probe, alphas=np.linspace(0, 2, 17), device="cuda")
fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(line.alpha, line.loss, marker="o", ms=3); ax.axvline(1, color="r", ls="--", lw=0.8); ax.set_xlabel("alpha (fraction of the step taken)"); ax.set_ylabel(f"loss on {PROBE_POINTS} points"); ax.set_title(f"step {STEP_FOCUS}: loss along the update", fontsize=9); ax.grid(alpha=0.25); plt.show()
abl = layerwise_update_ablation(tmpl, p, u, x_probe, y_probe, device="cuda"); abl["applied"] = abl.applied.str.replace("hidden_layers.0.weight", "W1").str.replace("hidden_layers.1.weight", "W2").str.replace("hidden_layers.2.weight", "W3").str.replace("output_layer.weight", "Wout")
print(abl.round(4).to_string(index=False))
# carried momentum vs fresh gradient: how much of this step's update was already in the optimizer's buffer?
gf = Gf[i]; beta = anat["beta1"]; kind = anat["optimizer"]
if kind == "sgd":                                  # update = -lr (beta * buffer + grad)
    mf = flatten_parameter_dict(anat["momentum_before"][i], names); lr = anat["lr"][i]; carried, fresh = lr * beta * mf, lr * gf
elif kind == "scalar_adam":                        # update = -lr (beta m_prev + (1 - beta) g) / (bc1 (sqrt(v_hat) + eps)) with one scalar v
    stt = anat["optimizer_state_before"][i]; mf = flatten_parameter_dict({nm: stt[nm]["exp_avg"] for nm in names}, names); lr = anat["lr"][i]
    scale = np.linalg.norm(Uf[i]) / max(np.linalg.norm(beta * mf + (1 - beta) * gf), 1e-30)        # = lr / (bc1 (sqrt(v_hat) + eps)), recovered from the update itself
    carried, fresh = scale * beta * mf, scale * (1 - beta) * gf
else:
    carried = fresh = None; print("AdamW: per-coordinate preconditioning, no scalar carried / fresh split")
if carried is not None:
    print(f"{kind}: |carried part| {np.linalg.norm(carried):.4f}, |fresh gradient part| {np.linalg.norm(fresh):.4f}, cos(carried, fresh) {float(carried @ fresh / max(np.linalg.norm(carried) * np.linalg.norm(fresh), 1e-30)):+.3f}, |update| {np.linalg.norm(Uf[i]):.4f}; SGD-equivalent lr of this step {anat['lr_sgd_equiv'][i]:.5f}")

In [ ]:
# After the peak: the first steps of the next interval (the catapult's way down). Needs the next snapshot to exist.
NEXT_STEPS = 40
nxt = replay_steps_full(sr, SPIKE_EPOCH_START + (epochs[1] - epochs[0]), step_from=0, step_to=NEXT_STEPS, data=data, device="cuda", probe_points=PROBE_POINTS)
Un = np.stack([flatten_parameter_dict(u, names) for u in nxt["updates"]])
lam = {s_: top_hessian_eigenvalue(tmpl, nxt["params_before"][s_], x_probe, y_probe, device="cuda", iters=25, mask=mask_np)["lambda_max"] * nxt["lr"][s_] for s_ in (0, 2, 5, 10, 20, NEXT_STEPS - 1)}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(nxt["step"], nxt["batch_loss"], lw=0.8, label="mini-batch loss"); axes[0].plot(nxt["step"], nxt["probe_loss"], lw=0.8, ls="--", label="probe loss"); axes[0].set_yscale("log"); axes[0].legend(fontsize=7)
axes[1].plot(nxt["step"], np.linalg.norm(Un, axis=1), lw=0.8, label="|update|"); ax2 = axes[1].twinx(); ax2.plot(list(lam), list(lam.values()), marker="o", ms=4, color="C3", label="lr * lambda_max"); ax2.axhline(2 * (1 + beta), color="r", ls="--", lw=0.8); ax2.set_ylabel("lr * lambda_max", color="C3"); axes[1].legend(fontsize=7, loc="upper left")
for ax in axes: ax.set_xlabel(f"step within the interval from epoch {nxt['epoch_start']}"); ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()
P0 = flatten_parameter_dict(anat["params_before"][0], names); P1 = flatten_parameter_dict(nxt["params_before"][-1], names)
print("net displacement from the start of the window to the end of the recovery:", round(float(np.linalg.norm(P1 - P0)), 3), {short(nm): round(float(np.linalg.norm(nxt["params_before"][-1][nm] - anat["params_before"][0][nm])), 3) for nm in names})